# DBIR-Style Incident Analysis

This notebook analyses a sample dataset of 50 VERIS-coded security incidents inspired by the Verizon Data Breach Investigations Report (DBIR).

We will reproduce several key DBIR-style findings:
- Actor type breakdown (external / internal / partner)
- Action category distribution (hacking, social, error, malware, misuse, …)
- Industry distribution
- Top attack vectors
- Organisation size vs. breach likelihood
- Breach vs. incident ratio and records exposed

**Dataset**: `../data/incidents.json` — 50 synthetic VERIS-coded incidents, European context, calendar year 2024.

In [ ]:
# Cell 2: Imports
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# Consistent style for all charts
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# Cell 3: Load incidents.json and normalise to a flat DataFrame

DATA_PATH = '../data/incidents.json'

with open(DATA_PATH) as f:
    incidents = json.load(f)

print(f'Total incidents loaded: {len(incidents)}')

# --- Build a flat DataFrame for easy analysis ---
rows = []
for inc in incidents:
    # Actor category (take first key)
    actor_keys = list(inc.get('actor', {}).keys())
    actor_cat = actor_keys[0] if actor_keys else 'unknown'

    # Action categories (may be multiple)
    action_cats = list(inc.get('action', {}).keys())

    # All attack vectors
    vectors = []
    for action_type, details in inc.get('action', {}).items():
        vectors.extend(details.get('vector', []))

    # Breach indicator
    disclosure = inc.get('attribute', {}).get('confidentiality', {}).get('data_disclosure', 'No')
    is_breach = disclosure == 'Yes'

    # Records exposed
    records = sum(
        item.get('amount', 0)
        for item in inc.get('attribute', {}).get('confidentiality', {}).get('data', [])
    )

    # Discovery time in hours
    disc = inc.get('timeline', {}).get('discovery', {})
    unit = disc.get('unit', '')
    value = disc.get('value', 0)
    if unit == 'Hours':
        discovery_hours = value
    elif unit == 'Days':
        discovery_hours = value * 24
    elif unit == 'Minutes':
        discovery_hours = value / 60
    else:
        discovery_hours = None

    # Country (first listed)
    countries = inc.get('victim', {}).get('country', ['Unknown'])
    country = countries[0] if countries else 'Unknown'

    rows.append({
        'incident_id': inc['incident_id'],
        'year': inc.get('timeline', {}).get('incident', {}).get('year'),
        'month': inc.get('timeline', {}).get('incident', {}).get('month'),
        'actor_cat': actor_cat,
        'action_cats': action_cats,
        'primary_action': action_cats[0] if action_cats else 'unknown',
        'vectors': vectors,
        'industry': inc.get('industry', 'Unknown'),
        'orgsize': inc.get('victim', {}).get('orgsize', 'Unknown'),
        'country': country,
        'is_breach': is_breach,
        'records_exposed': records,
        'discovery_hours': discovery_hours,
    })

df = pd.DataFrame(rows)
print(f'\nDataFrame shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Cell 4: Actor category breakdown

actor_counts = Counter(df['actor_cat'])
actor_labels = list(actor_counts.keys())
actor_values = list(actor_counts.values())

# Calculate percentages
total = sum(actor_values)
for label, val in sorted(actor_counts.items(), key=lambda x: -x[1]):
    print(f'  {label:<12} {val:>3}  ({val/total*100:.1f}%)')

# --- Chart ---
COLOR_MAP = {
    'external': '#ef5350',
    'internal': '#42a5f5',
    'partner':  '#66bb6a',
    'unknown':  '#bdbdbd',
}
colors = [COLOR_MAP.get(l, '#bdbdbd') for l in actor_labels]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(actor_labels, actor_values, color=colors, width=0.5, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, actor_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val}\n({val/total*100:.0f}%)', ha='center', va='bottom', fontsize=10)

ax.set_title('Incidents by Actor Category', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Actor Type')
ax.set_ylabel('Number of Incidents')
ax.set_ylim(0, max(actor_values) * 1.25)
plt.tight_layout()
plt.savefig('../data/actor_breakdown.png', dpi=100)
plt.show()
print('Chart saved to ../data/actor_breakdown.png')

In [ ]:
# Cell 5: Action category breakdown

# Explode multi-action incidents
all_actions = [action for actions in df['action_cats'] for action in actions]
action_counts = Counter(all_actions)

print('Action category counts (incidents may have multiple):')
for action, cnt in sorted(action_counts.items(), key=lambda x: -x[1]):
    print(f'  {action:<20} {cnt:>3}')

# --- Chart ---
ACTION_COLORS = {
    'hacking':     '#e53935',
    'social':      '#fb8c00',
    'error':       '#43a047',
    'malware':     '#8e24aa',
    'misuse':      '#1e88e5',
    'physical':    '#6d4c41',
    'environmental': '#546e7a',
    'unknown':     '#bdbdbd',
}

sorted_actions = sorted(action_counts.items(), key=lambda x: -x[1])
a_labels = [a[0] for a in sorted_actions]
a_values = [a[1] for a in sorted_actions]
a_colors = [ACTION_COLORS.get(l, '#bdbdbd') for l in a_labels]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(a_labels, a_values, color=a_colors, width=0.6, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, a_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=10)

ax.set_title('Action Categories (VERIS)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Action Category')
ax.set_ylabel('Count (multi-action incidents counted once per action)')
ax.set_ylim(0, max(a_values) * 1.2)
plt.tight_layout()
plt.savefig('../data/action_breakdown.png', dpi=100)
plt.show()
print('Chart saved to ../data/action_breakdown.png')

In [ ]:
# Cell 6: Industry distribution

# NAICS sector labels for readability
NAICS_LABELS = {
    '5221': 'Banking',
    '5241': 'Insurance',
    '8621': 'Hospitals',
    '6211': 'Medical offices',
    '6111': 'Education',
    '9211': 'Government',
    '4521': 'Retail (dept stores)',
    '4541': 'E-commerce',
    '5112': 'Software/IT',
    '5413': 'Professional services',
    '3345': 'Electronics mfg',
    '7211': 'Hotels/Travel',
    '5151': 'Media',
    '2381': 'Construction',
}

industry_counts = Counter(df['industry'])
# Map NAICS to labels
labeled = {NAICS_LABELS.get(k, f'NAICS {k}'): v
           for k, v in industry_counts.items()}
labeled_sorted = dict(sorted(labeled.items(), key=lambda x: -x[1]))

print('Incidents by industry:')
for ind, cnt in labeled_sorted.items():
    print(f'  {ind:<30} {cnt}')

# --- Pie chart ---
fig, ax = plt.subplots(figsize=(9, 6))
wedge_props = {'linewidth': 1, 'edgecolor': 'white'}
wedges, texts, autotexts = ax.pie(
    labeled_sorted.values(),
    labels=labeled_sorted.keys(),
    autopct=lambda p: f'{p:.1f}%' if p >= 4 else '',
    startangle=140,
    wedgeprops=wedge_props,
)
for text in texts:
    text.set_fontsize(9)
for autotext in autotexts:
    autotext.set_fontsize(8)
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title('Incidents by Industry (NAICS)', fontsize=13, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig('../data/industry_breakdown.png', dpi=100)
plt.show()
print('Chart saved to ../data/industry_breakdown.png')

In [ ]:
# Cell 7: Top attack vectors

all_vectors = [v for vectors in df['vectors'] for v in vectors]
vector_counts = Counter(all_vectors)

print('Top attack vectors:')
for vec, cnt in vector_counts.most_common(12):
    print(f'  {vec:<35} {cnt}')

# --- Horizontal bar chart (top 10) ---
top10 = dict(vector_counts.most_common(10))
v_labels = list(reversed(list(top10.keys())))
v_values = list(reversed(list(top10.values())))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(v_labels, v_values, color='#5c85d6', edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, v_values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)

ax.set_title('Top 10 Attack Vectors', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Count')
ax.set_xlim(0, max(v_values) * 1.2)
plt.tight_layout()
plt.savefig('../data/top_vectors.png', dpi=100)
plt.show()
print('Chart saved to ../data/top_vectors.png')

In [ ]:
# Cell 8: Organisation size vs. breach likelihood

ORGSIZE_LABELS = {'S': 'Small', 'M': 'Medium', 'L': 'Large'}

size_breach = df.groupby('orgsize')['is_breach'].agg(['sum', 'count'])
size_breach.columns = ['breaches', 'total']
size_breach['breach_rate'] = size_breach['breaches'] / size_breach['total']
size_breach.index = [ORGSIZE_LABELS.get(i, i) for i in size_breach.index]
size_breach = size_breach.reindex(['Small', 'Medium', 'Large'], fill_value=0)

print('Organisation size vs. breach likelihood:')
print(size_breach.to_string())

# --- Grouped bar chart ---
fig, ax1 = plt.subplots(figsize=(8, 5))

x = range(len(size_breach))
width = 0.35

bars_total = ax1.bar([i - width/2 for i in x], size_breach['total'],
                     width, label='Total Incidents', color='#90caf9', edgecolor='white')
bars_breach = ax1.bar([i + width/2 for i in x], size_breach['breaches'],
                      width, label='Confirmed Breaches', color='#ef9a9a', edgecolor='white')

ax1.set_xticks(list(x))
ax1.set_xticklabels(size_breach.index)
ax1.set_ylabel('Count')
ax1.set_title('Organisation Size vs. Breach Likelihood', fontsize=13, fontweight='bold', pad=12)
ax1.legend(loc='upper left')

# Add breach rate as text annotations
ax2 = ax1.twinx()
ax2.plot(list(x), size_breach['breach_rate'] * 100, 'D--',
         color='#e53935', markersize=8, linewidth=2, label='Breach Rate %')
ax2.set_ylabel('Breach Rate (%)', color='#e53935')
ax2.tick_params(axis='y', labelcolor='#e53935')
ax2.set_ylim(0, 120)
ax2.legend(loc='upper right')

for bar, val in zip(bars_breach, size_breach['breaches']):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../data/orgsize_breach.png', dpi=100)
plt.show()
print('Chart saved to ../data/orgsize_breach.png')

## Findings Summary

Based on the 50-incident sample dataset, the following DBIR-style findings emerge:

### Actor Patterns
- **~70% of incidents** involve **external** threat actors, consistent with DBIR trends showing financially motivated organised crime as the dominant actor type.
- **~20% internal**, primarily errors and misuse — insider threat remains a significant but under-detected risk.
- **~10% partner/vendor** — third-party risk materialises through both negligent misconfiguration and intentional actions.

### Attack Patterns
- **Hacking** (credential theft, vulnerability exploitation) is the most common action category.
- **Social engineering** (phishing, pretexting, BEC) is the leading initial access vector, consistent with DBIR finding that "humans are the most reliable attack surface."
- **Ransomware** continues to dominate availability incidents and accounts for significant operational disruption in SMEs.

### Industry & Organisation Size
- **Healthcare**, **government**, and **financial services** are disproportionately represented — aligning with DBIR findings on industries most targeted due to data value and regulatory visibility.
- **SMEs (small and medium organisations)** represent the majority of victims, consistent with the DBIR finding that smaller organisations face similar threat profiles to large ones but with far fewer defensive resources.

### Key Limitations
- Sample size: 50 incidents is insufficient for robust statistical inference. Real DBIR analyses use thousands of records.
- Selection bias: incidents in this dataset were synthetically generated to be representative; real VCDB data skews toward publicly disclosed incidents.
- Geographic scope: European context may differ from global DBIR findings in actor attribution and regulatory reporting patterns.

---
*Data source: Synthetic VERIS-coded incidents for educational use. European context, 2024.*

In [ ]:
# Cell 10: Export summary statistics to a dict (and optionally to JSON)

# Breach vs incident
breaches = df[df['is_breach'] == True]
breach_rate = len(breaches) / len(df)
total_records_exposed = breaches['records_exposed'].sum()

# External actor breaches
external_breaches = len(breaches[breaches['actor_cat'] == 'external'])
external_breach_pct = external_breaches / len(breaches) * 100 if len(breaches) > 0 else 0

# MTTD stats
discovery_times = df['discovery_hours'].dropna()
mttd_mean = discovery_times.mean()
mttd_median = discovery_times.median()

# Top action
top_action = action_counts.most_common(1)[0] if action_counts else ('unknown', 0)

# Top vector
top_vector = vector_counts.most_common(1)[0] if vector_counts else ('unknown', 0)

summary_stats = {
    'total_incidents': len(df),
    'confirmed_breaches': len(breaches),
    'breach_rate_pct': round(breach_rate * 100, 1),
    'total_records_exposed': int(total_records_exposed),
    'external_actor_pct': round(len(df[df['actor_cat'] == 'external']) / len(df) * 100, 1),
    'internal_actor_pct': round(len(df[df['actor_cat'] == 'internal']) / len(df) * 100, 1),
    'partner_actor_pct': round(len(df[df['actor_cat'] == 'partner']) / len(df) * 100, 1),
    'external_actor_in_breaches_pct': round(external_breach_pct, 1),
    'top_action_category': top_action[0],
    'top_action_count': top_action[1],
    'top_attack_vector': top_vector[0],
    'top_attack_vector_count': top_vector[1],
    'mttd_mean_hours': round(float(mttd_mean), 1),
    'mttd_median_hours': round(float(mttd_median), 1),
    'sme_pct': round(len(df[df['orgsize'].isin(['S', 'M'])]) / len(df) * 100, 1),
    'actor_breakdown': dict(actor_counts),
    'action_breakdown': dict(action_counts),
    'top_10_vectors': dict(vector_counts.most_common(10)),
}

# Pretty-print
print('=== DBIR-Style Summary Statistics ===')
for key, value in summary_stats.items():
    if not isinstance(value, dict):
        print(f'  {key:<40} {value}')

print()
print('Actor breakdown:',  summary_stats['actor_breakdown'])
print('Action breakdown:', summary_stats['action_breakdown'])
print('Top vectors:',      summary_stats['top_10_vectors'])

# Optionally save to JSON
import json as json_lib
with open('../data/summary_statistics.json', 'w') as f:
    json_lib.dump(summary_stats, f, indent=2)
print('\nSummary statistics saved to ../data/summary_statistics.json')